# StashFace — مطابقة على Kaggle (2x T4 GPU)

**قبل ما تشغل أي خلية:** من إعدادات الـnotebook (يمين الشاشة) اختار Accelerator = **GPU T4 x2**.

شغّل الخلايا بالترتيب من فوق لتحت.

In [1]:
# ============================================================
# خلية الإعداد — عدّل القيم دي بس، والباقي متلمسوش
# ============================================================

# رابط الـGitHub repo بتاع المشروع
GITHUB_REPO_URL = "https://github.com/kareemkamal10/stashface_pipeline"

# الـtoken بتاع حسابك على Hugging Face (لازم يكون عنده صلاحية Write)
HF_TOKEN = " "

# الـdataset اللي فيه ملف الأداء، وهيتحفظ فيه ناتج المطابقة كمان في الآخر
HF_DATASET_ID = "abdelwahabnabil500/datafile"

# اسم ملف بيانات الأداء جوه الـdataset
HF_INPUT_FILENAME = "tpdb_all_performers.json"

# اسم ملف النتيجة اللي هيتولّد، وهيترفع بنفس الاسم ده تاني للـdataset في الآخر
HF_OUTPUT_FILENAME = "matched_identities.json"

In [2]:
# ============================================================
# تحميل الكود وتثبيت المكتبات — مفيش حاجة تتعدل هنا
# ============================================================
import os

# منع hf CLI من سؤال "تحدّث دلوقتي؟" اللي بيعلّق جوه notebook (مفيش حد يرد عليه)
os.environ["HF_HUB_DISABLE_UPDATE_CHECK"] = "1"

# 1) تحميل الكود من الـGitHub repo
!git clone {GITHUB_REPO_URL} /kaggle/working/stashface_pipeline
%cd /kaggle/working/stashface_pipeline

# 2) تثبيت المكتبات المطلوبة
!pip install -q -r requirements.txt
!pip install -q -U "huggingface_hub[cli]>=1.13.0"

# 3) استبدال onnxruntime بنسخة الـGPU — عشان يستخدم الـT4 فعليًا مش الـCPU
!pip uninstall -y -q onnxruntime
!pip install -q "onnxruntime-gpu==1.26.0"  # pinned: onnxruntime-gpu>=1.27 defaults to CUDA 13, Kaggle T4 images still run CUDA 12.x

# 4) تسجيل الدخول لـHugging Face بالـtoken بتاعك
from huggingface_hub import login
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

Cloning into '/kaggle/working/stashface_pipeline'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 100 (delta 29), reused 38 (delta 16), pack-reused 47 (from 1)
Receiving objects: 100% (100/100), 96.89 MiB | 24.64 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/kaggle/working/stashface_pipeline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 70.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 66.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 36.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 77.4 MB/s eta 0:00:

In [3]:
# ============================================================
# تحميل بيانات المشروع (الموديل + قاعدة البيانات) + ملف الأداء
# ============================================================

# 5) تحميل بيانات المشروع (adaface model + performers.zvec) من الـbucket
!python setup.py --skip-install

# 6) تحميل ملف tpdb_all_performers.json من الـdataset بتاعك
from huggingface_hub import hf_hub_download
import shutil

local_path = hf_hub_download(
    repo_id=HF_DATASET_ID,
    repo_type="dataset",
    filename=HF_INPUT_FILENAME,
    token=HF_TOKEN,
)
shutil.copy(local_path, "tpdb_all_performers.json")
print("تم تحميل ملف البيانات:", local_path)

Clearing huggingface_hub cache at '/root/.cache/huggingface/' ...
Scanning remote bucket (16 files)
Scanning local directory (0 files)
Comparing files (16 paths)
Sync plan: hf://buckets/cc1234/stashface-data -> /kaggle/working/stashface_pipeline/data
  Uploads: 0
  Downloads: 16
  Deletes: 0
  Skips: 0
Syncing...

Sync completed.
Done. Bucket 'cc1234/stashface-data' synced into '/kaggle/working/stashface_pipeline/data/'.


tpdb_all_performers.json: reconstructing file:   0%|          |  0.00B / 27.0MB            

tpdb_all_performers.json: downloading bytes:           |  0.00B            

تم تحميل ملف البيانات: /root/.cache/huggingface/hub/datasets--abdelwahabnabil500--datafile/snapshots/30de015a36217a8e78a6faea6ef2ce5583a81ce4/tpdb_all_performers.json


## التشغيل الكامل على كل البيانات

دي هتاخد وقت طويل (ساعات، حسب حجم البيانات). لو الجلسة اتقفلت أو حصل أي كراش، رجّع شغّل نفس الخلية تاني — هيكمل من حيث ما وقف من غير ما يعيد اللي خلص منه.

In [4]:
!python run_kaggle.py

Loaded 114092 entries; 0 already matched previously; 114092 left to process.
Opening performers.zvec (shared across all GPU workers) ...
[GPU 0] loading AdaFace model ...
[GPU 0] ready.
[GPU 1] loading AdaFace model ...
[GPU 1] ready.
Starting: 8 batch(es) of up to 15000 entries each, 16 download workers, 2 GPU worker(s).
Batch 1/8: downloaded 15000/15000 in 1m58s.
100%|████████████████████████████████| 281857/281857 [00:10<00:00, 26060.04KB/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Batch 1/8: matched 15000 images in 5m35s (+8077 matches, 8077 total). [saved]
Batch 2/8: downloaded 15000/15000 in 2m41s.
Batch 2/8: matched 15000 images in 5m30s (+8083 matches, 16160 total). [saved]
Batch 3/8: downloaded 15000/15000 in 2m59s.
Batch 3/8: matched 15000 images in 5m32s (+7985 matches, 24145 total). [saved]
Batch 4/8: downloaded 15000/15000 in 3m03s.
Batch 4

## رفع ملف النتيجة (أهم خطوة)

شغّل الخلية دي بس بعد ما الخلية اللي فاتت تخلص تمامًا.

In [5]:
from huggingface_hub import HfApi

assert os.path.exists("matched_identities.json"), "الملف مش موجود — تأكد إن خلية التشغيل الكامل خلصت من غير أخطاء"

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="matched_identities.json",
    path_in_repo=HF_OUTPUT_FILENAME,
    repo_id=HF_DATASET_ID,
    repo_type="dataset",
    token=HF_TOKEN,
    commit_message="Add matched_identities.json from stashface pipeline run",
)
print("تم رفع ملف النتائج بنجاح إلى:", HF_DATASET_ID)

تم رفع ملف النتائج بنجاح إلى: abdelwahabnabil500/datafile
